In [1]:
library(openxlsx)
library(HGNChelper)

Please cite our software :) 
 
 Sehyun Oh et al. HGNChelper: identification and correction of invalid gene symbols for human and mouse. F1000Research 2020, 9:1493. DOI: https://doi.org/10.12688/f1000research.28033.1 
 
 Type `citation('HGNChelper')` for a BibTeX entry.



In [2]:

# load gene set preparation function
source("https://raw.githubusercontent.com/IanevskiAleksandr/sc-type/master/R/gene_sets_prepare.R")
# load cell type annotation function
source("https://raw.githubusercontent.com/IanevskiAleksandr/sc-type/master/R/sctype_score_.R")

# DB file
db_ <- "https://raw.githubusercontent.com/IanevskiAleksandr/sc-type/master/ScTypeDB_full.xlsx";
tissue <- "Brain" # e.g. Immune system,Pancreas,Liver,Eye,Kidney,Brain,Lung,Adrenal,Heart,Intestine,Muscle,Placenta,Spleen,Stomach,Thymus 

# prepare gene sets
gs_list <- gene_sets_prepare(db_, tissue)

In [3]:
library(dplyr)
library(tidyverse)
library(data.table)
library(ggplot2)
library(RColorBrewer)
#library(genefilter)
#library(irlba)

write.table_hh_tab<-function(X,y,z){###########名前付きデータフレームを左上つきで出力する自作関数yが左上zが出力名
ZZZ<-cbind(rownames(X),X)
if(is.null(rownames(X))=="TRUE"){
print("行名と列名が必要です")
}else{
colnames(ZZZ)[1]<-y
write.table(ZZZ,z,col.names=TRUE,row.names=FALSE,quote=FALSE,sep="\t")
}
}
#library("edgeR")

library(Seurat)

options(stringsAsFactors=F)
options(scipen=100)　　#指数表記を回避


次のパッケージを付け加えます: ‘dplyr’


以下のオブジェクトは ‘package:stats’ からマスクされています:

    filter, lag


以下のオブジェクトは ‘package:base’ からマスクされています:

    intersect, setdiff, setequal, union


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ forcats   1.0.0     ✔ readr     2.1.5
✔ ggplot2   3.5.1     ✔ stringr   1.5.1
✔ lubridate 1.9.3     ✔ tibble    3.2.1
✔ purrr     1.0.2     ✔ tidyr     1.3.1
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

次のパッケージを付け加えます: ‘data.table’


以下のオブジェクトは ‘package:lubridate’ からマスクされています:

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


以下のオブジェクトは ‘package:purrr’ からマスクされています:

    transpose


以下のオブジェクトは ‘package:dplyr’ からマスクされています:

    between, first, last


要求されたパッケージ SeuratObject をロード中です

要求されたパッケージ

In [ ]:
data=readRDS("~/scRNA-seq_data/result/GSE163018_gene_filter_only/GSM4970298_H28126_after_norm.rds")
data.markers <- FindAllMarkers(data, only.pos = TRUE)
# check Seurat object version (scRNA-seq matrix extracted differently in Seurat v4/v5)
seurat_package_v5 <- isFALSE('counts' %in% names(attributes(data[["RNA"]])));
print(sprintf("Seurat object %s is used", ifelse(seurat_package_v5, "v5", "v4")))
# extract scaled scRNA-seq matrix
scRNAseqData_scaled <- if (seurat_package_v5) as.matrix(data[["RNA"]]$scale.data) else as.matrix(data[["RNA"]]@scale.data)


In [ ]:

# run ScType
es.max <- sctype_score(scRNAseqData = scRNAseqData_scaled, scaled = TRUE, gs = gs_list$gs_positive, gs2 = gs_list$gs_negative)

# merge by cluster
cL_resutls <- do.call("rbind", lapply(unique(data@meta.data$seurat_clusters), function(cl){
    es.max.cl = sort(rowSums(es.max[ ,rownames(data@meta.data[data@meta.data$seurat_clusters==cl, ])]), decreasing = !0)
    head(data.frame(cluster = cl, type = names(es.max.cl), scores = es.max.cl, ncells = sum(data@meta.data$seurat_clusters==cl)), 10)
}))
sctype_scores <- cL_resutls %>% group_by(cluster) %>% top_n(n = 1, wt = scores)  

# set low-confident (low ScType score) clusters to "unknown"
sctype_scores$type[as.numeric(as.character(sctype_scores$scores)) < sctype_scores$ncells/4] <- "Unknown"
print(sctype_scores[,1:3])

In [ ]:
write.table(sctype_scores[,1:3], "~/scRNA-seq_data/result/GSE163018_gene_filter_only_ver2/GSM4970298_H28126_after_norm_scType_cluster.csv",row.names=FALSE,col.names=TRUE, sep=",", append=F, quote=F)

In [ ]:
data@meta.data$sctype_classification = ""
for(j in unique(sctype_scores$cluster)){
  cl_type = sctype_scores[sctype_scores$cluster==j,]; 
  data@meta.data$sctype_classification[data@meta.data$seurat_clusters == j] = as.character(cl_type$type[1])
}

DimPlot(data, reduction = "umap", label = TRUE, repel = TRUE, group.by = 'sctype_classification')    

In [ ]:
library(ggrepel)

# Assign cell type classifications to metadata
data@meta.data$sctype_classification <- ""
for (j in unique(sctype_scores$cluster)) {
  cl_type <- sctype_scores[sctype_scores$cluster == j, ]
  data@meta.data$sctype_classification[data@meta.data$seurat_clusters == j] <- 
    as.character(cl_type$type[1])
}

# Generate UMAP visualization

p <- DimPlot(data, 
             reduction = "umap", 
             label = FALSE, 
             repel = TRUE, 
             group.by = "sctype_classification",
             pt.size = 1) +
  theme_classic() +
  theme(
    axis.text = element_text(size = 16, color = "black"),
    axis.title = element_text(size = 16, color = "black"),
    legend.position = "none",
    text = element_text(family = "Helvetica")
  )


library(dplyr)
umap_coords <- as.data.frame(Embeddings(data, "umap"))
umap_coords$cell_type <- data@meta.data$sctype_classification
label_pos <- umap_coords %>%
  group_by(cell_type) %>%
  summarise(UMAP_1 = median(umap_1), UMAP_2 = median(umap_2))

label_pos <- label_pos %>%
  mutate(
    UMAP_2 = ifelse(cell_type == "Astrocytes", UMAP_2+0.3, UMAP_2),
    UMAP_1 = ifelse(cell_type == "Neural Progenitor cells", UMAP_1 + 1.5, UMAP_1),
    UMAP_2 = ifelse(cell_type == "Neural Progenitor cells", UMAP_2 + 1.0, UMAP_2),
    UMAP_2 = ifelse(cell_type == "Neural stem cells", UMAP_2 + 0.5, UMAP_2)
  )


p <- p + geom_text_repel(data = label_pos, 
                          aes(x = UMAP_1, y = UMAP_2, label = cell_type),
                          size = 6, 
                          color = "black", 
                          fontface = "bold",
                          family = "Helvetica",
                          box.padding = 1.0,
                          point.padding = 0.5,
                          max.overlaps = Inf,
                          min.segment.length = 0,
                          segment.color = NA,
                          inherit.aes = FALSE)


 #Save high-resolution figure
ggsave("~/scRNA-seq_data/result/GSE163018_gene_filter_only_ver2/plots2/umap_cell_types_ver3.png", 
       plot = p, 
       width = 7, 
       height = 7, 
       units = "in", 
       dpi = 300)

In [ ]:
p

In [ ]:
umap_coords

# UMAP of CC2top, CC3 bottom

In [ ]:
CC2_df=read.table("~/scRNA-seq_data/result/GSE163018_gene_filter_only_ver2/transcriptome_raw_CC/CC2_rotation_CCscore.txt")

In [ ]:
colnames(CC2_df)=c("id", "CCscore")

In [ ]:


threshold <- quantile(CC2_df$CCscore, 0.9)

CC2_df2=CC2_df %>% mutate(CC2_top=ifelse(CCscore>threshold, TRUE, FALSE))

low_threshold <- quantile(CC2_df$CCscore, 0.1)

CC2_df3=CC2_df2 %>% mutate(CC2_bottom=ifelse(CCscore<low_threshold, TRUE, FALSE))


In [ ]:
rownames(CC2_df3)=CC2_df3$id
CC2_df4=CC2_df3 %>% select(-c("id"))

In [ ]:
CC2_matrix=as.matrix(CC2_df4)

In [ ]:
data=AddMetaData(object = data, metadata = CC2_matrix)

In [ ]:
# Generate UMAP visualization
p <- DimPlot(data, 
             reduction = "umap", 
             label = FALSE,  
             group.by = "CC2_top",
             pt.size = 0.5) +
  scale_color_manual(values = c("0" = "grey", "1" = "#DC0000FF")) +
  theme_classic() +
  theme(
    axis.text = element_text(size = 16, color = "black"),
    axis.title = element_text(size = 16, color = "black"),
    legend.position = "none"
  )


In [ ]:
p

In [ ]:
# Save high-resolution figure
ggsave("/home/imgtaka/scRNA-seq_data/result/GSE163018_gene_filter_only_ver2/plots2/CC2_top_binary_ver2.pdf", 
       plot = p, 
       width = 7, 
       height = 7, 
       units = "in", 
       dpi = 300)

In [ ]:
CC3_df=read.table("/home/imgtaka/scRNA-seq_data/result/GSE163018_gene_filter_only_ver2/transcriptome_raw_CC/CC3_rotation_CCscore.txt")
colnames(CC3_df)=c("id", "CCscore")

threshold <- quantile(CC3_df$CCscore, 0.9)

CC3_df2=CC3_df %>% mutate(CC3_top=ifelse(CCscore>threshold, TRUE, FALSE))

low_threshold <- quantile(CC3_df$CCscore, 0.1)

CC3_df3=CC3_df2 %>% mutate(CC3_bottom=ifelse(CCscore<low_threshold, TRUE, FALSE))
rownames(CC3_df3)=CC3_df3$id
CC3_df4=CC3_df3 %>% select(-c("id"))
CC3_matrix=as.matrix(CC3_df4)
data=AddMetaData(object = data, metadata = CC3_matrix)
# Generate UMAP visualization
p <- DimPlot(data, 
             reduction = "umap", 
             label = FALSE,  
             group.by = "CC3_bottom",
             pt.size = 0.5) +
  scale_color_manual(values = c("0" = "grey", "1" = "#DC0000FF")) +
  theme_classic() +
  theme(
    axis.text = element_text(size = 16, color = "black"),
    axis.title = element_text(size = 16, color = "black"),
    legend.position = "none"
  )


In [ ]:
p

In [ ]:
# Save high-resolution figure
ggsave("/home/imgtaka/scRNA-seq_data/result/GSE163018_gene_filter_only_ver2/plots2/CC3_bottom_binary_ver2.pdf", 
       plot = p, 
       width = 7, 
       height = 7, 
       units = "in", 
       dpi = 300)

sessionInfo()

R version 4.4.2 (2024-10-31)
Platform: x86_64-conda-linux-gnu
Running under: Red Hat Enterprise Linux 8.8 (Ootpa)

Matrix products: default
BLAS/LAPACK: /hshare1/ZETTAI_path_WA_slash_home_KARA/home/imgtaka/miniconda3/envs/Seurat/lib/libopenblasp-r0.3.28.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=ja_JP.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=ja_JP.UTF-8        LC_COLLATE=ja_JP.UTF-8    
 [5] LC_MONETARY=ja_JP.UTF-8    LC_MESSAGES=ja_JP.UTF-8   
 [7] LC_PAPER=ja_JP.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=ja_JP.UTF-8 LC_IDENTIFICATION=C       

time zone: Asia/Tokyo
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
 [1] Seurat_5.1.0       SeuratObject_5.0.2 sp_2.1-4           RColorBrewer_1.1-3
 [5] data.table_1.16.4  lubridate_1.9.3    forcats_1.0.0      stringr_1.5.1     
 [9] purrr_1.0.2        readr_2.1.5        tidyr_1.3.1        tibble_3.2.1      
[13] ggplot2_3.5.1      tidyverse_2.0.0    dplyr_1.1.4        HGNChelper_0.8.15 
[17] openxlsx_4.2.8    

loaded via a namespace (and not attached):
  [1] jsonlite_1.8.9         magrittr_2.0.3         spatstat.utils_3.1-1  
  [4] farver_2.1.2           vctrs_0.6.5            ROCR_1.0-11           
  [7] spatstat.explore_3.3-3 base64enc_0.1-3        htmltools_0.5.8.1     
 [10] sctransform_0.4.1      parallelly_1.40.1      KernSmooth_2.23-24    
 [13] htmlwidgets_1.6.4      ica_1.0-3              plyr_1.8.9            
 [16] plotly_4.10.4          zoo_1.8-12             uuid_1.2-1            
 [19] igraph_2.0.3           mime_0.12              lifecycle_1.0.4       
 [22] pkgconfig_2.0.3        Matrix_1.6-5           R6_2.5.1              
 [25] fastmap_1.2.0          fitdistrplus_1.2-1     future_1.34.0         
 [28] shiny_1.9.1            digest_0.6.37          colorspace_2.1-1      
 [31] patchwork_1.3.0        tensor_1.5             RSpectra_0.16-2       
 [34] irlba_2.3.5.1          progressr_0.15.1       fansi_1.0.6           
 [37] spatstat.sparse_3.1-0  timechange_0.3.0       httr_1.4.7            
 [40] polyclip_1.10-7        abind_1.4-8            compiler_4.4.2        
 [43] withr_3.0.2            fastDummies_1.7.4      MASS_7.3-61           
 [46] tools_4.4.2            splitstackshape_1.4.8  lmtest_0.9-40         
 [49] zip_2.3.1              httpuv_1.6.15          future.apply_1.11.3   
 [52] goftest_1.2-3          glue_1.8.0             nlme_3.1-166          
 [55] promises_1.3.2         grid_4.4.2             pbdZMQ_0.3-13         
 [58] Rtsne_0.17             cluster_2.1.6          reshape2_1.4.4        
 [61] generics_0.1.3         gtable_0.3.6           spatstat.data_3.1-4   
 [64] tzdb_0.4.0             hms_1.1.3              utf8_1.2.4            
 [67] spatstat.geom_3.3-4    RcppAnnoy_0.0.22       ggrepel_0.9.6         
 [70] RANN_2.6.2             pillar_1.9.0           spam_2.11-0           
 [73] IRdisplay_1.1          RcppHNSW_0.6.0         later_1.4.1           
 [76] splines_4.4.2          lattice_0.22-6         survival_3.7-0        
 [79] deldir_2.0-4           tidyselect_1.2.1       miniUI_0.1.1.1        
 [82] pbapply_1.7-2          gridExtra_2.3          scattermore_1.2       
 [85] matrixStats_1.4.1      stringi_1.8.4          lazyeval_0.2.2        
 [88] evaluate_1.0.1         codetools_0.2-20       cli_3.6.3             
 [91] uwot_0.2.2             IRkernel_1.3.2         xtable_1.8-4          
 [94] reticulate_1.40.0      repr_1.1.7             munsell_0.5.1         
 [97] Rcpp_1.0.13-1          globals_0.16.3         spatstat.random_3.3-2 
[100] png_0.1-8              spatstat.univar_3.1-1  parallel_4.4.2        
[103] dotCall64_1.2          listenv_0.9.1          viridisLite_0.4.2     
[106] scales_1.3.0           ggridges_0.5.6         leiden_0.4.3.1        
[109] crayon_1.5.3           rlang_1.1.4            cowplot_1.1.3         

Click to add a cell.